# Text embeddings from scratch

The extension of the happy and sad notebook turns a sentence into $768$ numbers with BERT, and one linear layer picks a face from them:

https://colab.research.google.com/github/eth-bmai-hs26/w4-cx-public/blob/main/happy-sad-diffusion/happy_sad_diffusion_extension.ipynb

BERT arrives already trained. This notebook builds a tiny text embedder of its own, from nothing but the same thirty sentences, and trains it in seconds:

* every **word** gets **two numbers** of its own, its **embedding**, kept in a table;
* a **sentence** gets the **average** of the embeddings of its words;
* a **wide network** turns that average into **three numbers**, the embedding of the sentence. Training pulls them towards the one-hot of the sentence's expression, the prompts of the extension: $[1, 0, 0]$ for happy, $[0, 1, 0]$ for sad, $[0, 0, 1]$ for surprised.

Training finds all of these numbers at once, the two numbers of every word and the weights of the network. Then we look at where the words landed, and at the three numbers of every sentence as a point in space.

**Your part: three tasks, each marked with 🎯, six lines in all.** The two layers of the wide network and the average of the words' numbers in part 3, and the three calls of one training step in part 4. Everything else is done for you. The cells that show only a title load the libraries and draw the pictures: run them, there is nothing to change inside. Their Show code button opens one if you are curious. Run the cells from top to bottom, then try your own sentences at the end.

**Stuck?** Every task has a hint below it: click it to open it. And this short worked example trains a small network on made-up ice cream sales, with the same kinds of lines as tasks 1 and 3, and all of its code shown:

https://colab.research.google.com/github/eth-bmai-hs26/w4-cx-public/blob/main/pufferfish-autoencoder/pytorch_example.ipynb

In [1]:
#@title Imports { display-mode: "form" }
import torch
from torch import nn
import matplotlib.pyplot as plt
import plotly.graph_objects as go

## 1. Thirty sentences

The thirty short descriptions of the extension, ten for each expression, made up for that notebook. Each sentence comes with its expression: $0$ for happy, $1$ for sad, $2$ for surprised.

In [2]:
HAPPY_TEXTS = ["a happy face", "a smiling face", "someone who is glad", "a cheerful person", "she is delighted",
               "he is laughing", "a big smile", "a pleased child", "they look very happy", "a grinning man"]
SAD_TEXTS = ["a sad face", "an unhappy face", "someone who is crying", "a gloomy person", "she is upset",
             "he is miserable", "tears running down a face", "a heartbroken child", "they look very sad",
             "a tearful woman"]
SURPRISED_TEXTS = ["a surprised face", "an astonished face", "someone who is amazed", "a shocked person",
                   "she is stunned", "he is startled", "a mouth wide open in surprise", "a speechless child",
                   "they look very surprised", "an open-mouthed man"]
EXPRESSIONS = ["happy", "sad", "surprised"]

sentences = HAPPY_TEXTS + SAD_TEXTS + SURPRISED_TEXTS            # 30 sentences
labels = torch.tensor([0] * 10 + [1] * 10 + [2] * 10)            # their expressions: 0 happy, 1 sad, 2 surprised
print(len(sentences), "sentences, for example:", sentences[0], "/", sentences[10], "/", sentences[20])

30 sentences, for example: a happy face / a sad face / a surprised face


## 2. The words

The **vocabulary** is the list of the different words of the thirty sentences, in alphabetical order. A word is known by its place in the list: the first word is number $0$, the second number $1$, and so on.

In [3]:
vocabulary = sorted(set(" ".join(sentences).split()))            # every different word, once
print(len(vocabulary), "words:", vocabulary)
print('"smiling" is word number', vocabulary.index("smiling"))

49 words: ['a', 'amazed', 'an', 'astonished', 'big', 'cheerful', 'child', 'crying', 'delighted', 'down', 'face', 'glad', 'gloomy', 'grinning', 'happy', 'he', 'heartbroken', 'in', 'is', 'laughing', 'look', 'man', 'miserable', 'mouth', 'open', 'open-mouthed', 'person', 'pleased', 'running', 'sad', 'she', 'shocked', 'smile', 'smiling', 'someone', 'speechless', 'startled', 'stunned', 'surprise', 'surprised', 'tearful', 'tears', 'they', 'unhappy', 'upset', 'very', 'who', 'wide', 'woman']
"smiling" is word number 33


## 3. The model

Three pieces.

* `nn.Embedding` is a **table** with one row for each word of the vocabulary and two numbers in each row, the **embedding** of the word. Its numbers are weights like the weights of the network, so training moves them. They start as small random numbers, with a spread of $0.02$, as BERT's own table did before its training, so every word starts close to $(0, 0)$.
* `three_numbers(sentence)` looks up the row of every word of the sentence and takes the **average** of those rows: two numbers for the whole sentence.
* The **wide network**, one hidden layer of $100$ neurons as in the universal approximation theorem of weekend 1, turns those two numbers into the **three numbers** of the sentence. The biggest of the three is the network's answer.

🎯 **Task 1.** Write the two layers of the wide network, one line each:

* the first takes the $2$ numbers of the average to the $100$ neurons of the hidden layer, followed by an `nn.ReLU()`,
* the second takes those $100$ numbers to the $3$ numbers of the sentence, with no `nn.ReLU()` after it.

<details>
<summary><b>Hint for task 1</b> (click to open)</summary>

A line is one layer, `nn.Linear(numbers in, numbers out)`, and the numbers out of one line are the numbers in of the next. The ice cream example writes its network the same way, in its part 2: its first line, `nn.Linear(1, 32), nn.ReLU(),`, takes $1$ number to $32$, and its last line, `nn.Linear(32, 1),`, has no `nn.ReLU()`. Here the first line takes $2$ numbers to $100$, and the last line takes $100$ numbers to $3$.

</details>

🎯 **Task 2.** Complete `three_numbers`. Its first two lines look up the words of the sentence in the table: `vectors` holds one row of $2$ numbers for each word, $3$ rows for "a happy face". Write `average`, the average of those rows: $2$ numbers, the average of their first numbers and the average of their second numbers.

<details>
<summary><b>Hint for task 2</b> (click to open)</summary>

`vectors.mean()` would average all the numbers of `vectors` into one single number. `vectors.mean(dim=0)` averages down the rows instead, column by column: one average for each column, the $2$ numbers asked for here.

</details>

In [ ]:
torch.manual_seed(0)   # seeded: the same starting numbers on every computer

embedding = nn.Embedding(len(vocabulary), 2)                     # a table: 2 numbers for every word
nn.init.normal_(embedding.weight, std=0.02)                      # every word starts close to (0, 0)
start = embedding.weight.detach().clone()                        # kept, to see later how far each word moved

# ============================================================
# 🎯 TASK 1: the two layers of the wide network
# ============================================================
network = nn.Sequential(
    # 🎯 COMPLETE THESE TWO LINES: an nn.Linear each, and an nn.ReLU() after the first
    nn.Lineaer(2, 100), nn.ReLU(),    # the hidden layer: 2 numbers in, 100 out
    nn.Lineaer(100, 3),    # the last layer: 100 numbers in, 3 out
)


# ============================================================
# 🎯 TASK 2: the average, in three_numbers
# ============================================================
def three_numbers(sentence):
    ids = torch.tensor([vocabulary.index(word) for word in sentence.split()])   # the number of every word
    vectors = embedding(ids)                                     # its row of the table: 2 numbers per word
    # 🎯 COMPLETE THIS LINE: the average of the rows of vectors, 2 numbers for the sentence
    average = vectors.mean(dim=0)
    return network(average)                                      # 3 numbers for the sentence


print(three_numbers("a happy face"))                             # before training they mean nothing yet

## 4. Training

The loop of weekend 1: while the loss is too big, take a step. `torch.eye(3)` holds the three one-hots, one per row, and `targets` picks the one-hot of every sentence's expression. A step computes the three numbers of all thirty sentences, and the loss is their average squared difference from the targets, the squared miss of the house prices of weekend 1. The optimizer moves the weights of the network and the two numbers of every word in the table. The loop stops once the loss is below $0.01$, a typical miss of $0.1$, after a few dozen steps.

🎯 **Task 3.** One training step is three calls, always in this order:

1. The optimizer forgets the gradients of the previous step.
2. The loss computes its gradient, backwards through the network and the table.
3. The optimizer moves every weight, and the two numbers of every word, a small step against its gradient.

<details>
<summary><b>Hint for task 3</b> (click to open)</summary>

The loop of the ice cream example, in its part 3, ends with the same three calls, in the same order: copy them.

</details>

On Colab the loop stopped after about $7$ seconds. If the loss printed under the cell is still above $0.1$ after half a minute, stop the cell with the square button next to it and check your lines.

In [ ]:
# ============================================================
# 🎯 TASK 3: one training step, in the loop below
# ============================================================
targets = torch.eye(3)[labels]                                   # the one-hot of every sentence's expression
optimizer = torch.optim.Adam(list(embedding.parameters()) + list(network.parameters()), lr=0.01)

loss = 1.0                                                       # any start above the threshold
while loss > 0.01:
    numbers = torch.stack([three_numbers(sentence) for sentence in sentences])   # 30 sentences, 3 numbers each
    loss = ((numbers - targets) ** 2).mean()                     # the average squared difference
    # 🎯 COMPLETE THESE THREE LINES: one training step
    ____    # 1. the optimizer forgets the gradients of the previous step
    ____    # 2. the loss computes its gradient, backwards through the network and the table
    ____    # 3. the optimizer moves every weight, and every word, a small step against its gradient
    print(f"\rloss: {loss.item():.4f}", end="")

## 5. Where the words landed

A few words, with their two numbers before and after training.

In [ ]:
for word in ["happy", "smiling", "sad", "crying", "surprised", "shocked", "a", "is", "face"]:
    row = vocabulary.index(word)
    print(f"{word:>9}: before {start[row].numpy().round(2)}, after {embedding.weight[row].detach().numpy().round(2)}")

And every word as a point: its first number across, its second number up. The colours are not something the network saw. They say in which sentences a word appears: only in happy ones (vermillion dots), only in sad ones (blue squares), only in surprised ones (black diamonds), or in sentences of more than one expression (grey).

In [ ]:
#@title Every word as a point { display-mode: "form" }
COLOURS = ["#D55E00", "#0072B2", "#000000"]          # happy vermillion, sad blue, surprised black
MARKERS = ["o", "s", "D"]                            # and a shape for each: dot, square, diamond
GREY = "#999999"                                     # words that appear with more than one expression

points = embedding.weight.detach()
fig, ax = plt.subplots(figsize=(12, 8.5))
letter = 0.012 * (points[:, 0].max() - points[:, 0].min()).item()   # about the width of a letter
line = 0.035 * (points[:, 1].max() - points[:, 1].min()).item()     # about the height of a line
taken = [(x - letter / 2, x + letter / 2, y) for x, y in points.tolist()]   # every point, then every label
lowest = points[:, 1].min().item()
for row in points[:, 1].argsort(descending=True).tolist():          # the words from the top down
    word = vocabulary[row]
    appears = sorted(set(labels[[word in sentence.split() for sentence in sentences]].tolist()))
    colour, marker = (COLOURS[appears[0]], MARKERS[appears[0]]) if len(appears) == 1 else (GREY, ".")
    x, y = points[row].tolist()
    ax.plot(x, y, marker, color=colour, markersize=7)
    left, right, label_y = x + letter, x + letter * (len(word) + 1.5), y
    while any(l < right and left < r and abs(label_y - ly) < line for l, r, ly in taken):
        label_y = label_y - line                                     # one line lower, until it covers nothing
    taken.append((left, right, label_y))
    lowest = min(lowest, label_y)
    if label_y != y:
        ax.plot([x, left], [y, label_y], color=colour, linewidth=0.5)   # a thin line from the label to its point
    ax.text(left, label_y, word, color=colour, fontsize=11, va="center")
ax.set_ylim(bottom=lowest - line)
ax.set(xlabel="first number", ylabel="second number", title="The two numbers of every word, after training")
plt.show()

What to look for:

* The words of each expression gathered in a group of their own, away from the middle, where every word started.
* The grey words appear with more than one expression, `a`, `is`, `face`, `someone`. They say little about the expression, so training moved them much less.
* Words that appear in exactly the same sentences get the same push at every step, so they land next to each other: `big` and `smile` appear only in "a big smile", `mouth`, `wide`, `open`, `in` and `surprise` only in "a mouth wide open in surprise". `woman` appears only in "a tearful woman", so it sits next to `tearful`, among the sad words. The table knows a word only through the sentences it appears in.

## 6. Every sentence as a point in space

The three numbers of each of the thirty sentences, as a point: the happy number, the sad number and the surprised number. Training pulled every sentence towards the one-hot of its expression, the corners $[1, 0, 0]$, $[0, 1, 0]$ and $[0, 0, 1]$, and the sentences of each expression form a **cluster** of their own. Drag the picture to turn it, and point at a dot to read its sentence.

In [ ]:
#@title The thirty sentences in space { display-mode: "form" }
SYMBOLS = ["circle", "square", "diamond"]            # the same shapes as in the picture of the words
HOVER = "%{text}<br>happy %{x:.2f}<br>sad %{y:.2f}<br>surprised %{z:.2f}<extra></extra>"
# A plotly 3D picture keeps the mouse wheel for itself, even with scrollZoom off, so the page would stop
# scrolling under the pointer. This line, run in the browser after drawing, hands the wheel back to the page.
FREE_WHEEL = ("var camera = document.getElementById('{plot_id}')._fullLayout.scene._scene.camera;"
              "camera.element.removeEventListener('wheel', camera.wheelListener);")


def draw_sentences(new_sentences=()):
    fig = go.Figure()
    with torch.no_grad():
        numbers = torch.stack([three_numbers(sentence) for sentence in sentences]).numpy()
    for k, name in enumerate(EXPRESSIONS):
        mine = (labels == k).numpy()
        fig.add_trace(go.Scatter3d(x=numbers[mine, 0], y=numbers[mine, 1], z=numbers[mine, 2],
                                   mode="markers", name=name, hovertemplate=HOVER,
                                   text=[sentence for sentence, m in zip(sentences, mine) if m],
                                   marker=dict(color=COLOURS[k], symbol=SYMBOLS[k], size=5)))
    readable, points = [], []
    for sentence in new_sentences:
        words = sentence.lower().split()
        known = [word for word in words if word in vocabulary]          # the words it can read
        unknown = [word for word in words if word not in vocabulary]    # the words it never saw, left out
        if known:
            with torch.no_grad():
                point = three_numbers(" ".join(known))
            readable.append(sentence)
            points.append(point)
            print(f"{sentence!r:30} {EXPRESSIONS[point.argmax()]:>9}  {point.numpy().round(2)}   never saw: {unknown}")
        else:
            print(f"{sentence!r:30} it never saw any of these words")
    if points:
        new = torch.stack(points).numpy()
        fig.add_trace(go.Scatter3d(x=new[:, 0], y=new[:, 1], z=new[:, 2], mode="markers+text",
                                   name="new sentences", text=readable, hovertemplate=HOVER,
                                   marker=dict(color=GREY, symbol="x", size=5)))
    fig.update_layout(scene=dict(xaxis_title="happy", yaxis_title="sad", zaxis_title="surprised"),
                      template="plotly_white", height=650, margin=dict(l=0, r=0, t=0, b=0))
    fig.show(config={"scrollZoom": False}, post_script=FREE_WHEEL)   # the wheel scrolls; drag to turn the picture


draw_sentences()

## 7. Try it

New sentences, which the network never saw. It reads only the words of its vocabulary, the list of part 2, and leaves every other word out. The cell prints the answer and the three numbers of each sentence, and draws it as a grey cross among the thirty.

* Write your own sentences in the list and run the cell again.
* Where do "a smiling and crying face" and "a smiling and shocked face" land, compared with the clusters?
* "tears of joy" and "a face that is not happy": which words does the network read, and what does it make of them?
* Compare "a smiling man" and "a smiling woman". Why do they land in different places?

In [ ]:
new_sentences = ["a smiling child", "he is upset", "she is amazed",
                 "a smiling and crying face", "tears of joy", "a face that is not happy"]   # write your own here
draw_sentences(new_sentences)

## 8. What BERT does differently

The same idea, a table of numbers for every token and a network on top, at a much larger scale.

* BERT's table has a row for each of $30{,}522$ tokens, words and pieces of words, so it has numbers for almost any English text. Ours has $49$ rows.
* Each row has $768$ numbers instead of $2$, learned from billions of words of Wikipedia and books instead of thirty sentences.
* BERT does not simply average. Its attention layers, the transformers of weekend 2, let the numbers of every word depend on the words around it, so the order of the words counts. Our average gives the same point to the same words in any order.
* BERT was trained to guess hidden words of its texts, and whether one sentence follows another, tasks that need no labels, which is why its numbers serve many jobs. Ours learned one job: telling three expressions apart.